# CME Futures: Stochastic Discount-Factor Features

The stochastic discount-factor model learns fold-scoped latent factors from the product panel and
maps them to each declared forward-return horizon. Training rows determine the representation;
validation rows are transformed without refitting. The fitted model, fold identity, prediction
shard, and eligible validation keys are persisted together.

The notebook executes the declared SDF configurations and publishes their catalog rows. IC remains
diagnostic. The equal-weight validation backtest in `13_backtest` selects configurations.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [ ]:
"""Fit the declared CME futures stochastic discount-factor population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [ ]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

Both return horizons use the named stochastic discount-factor configuration. The resolved plan
shows the eligible rows, folds, feature count, checkpoint schedule, and identity before fitting.

In [ ]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog(
    "latent_factors",
    labels=ALL_LABELS,
    config_names=("sdf",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

In [ ]:
resolved_model_plan(resolved)

## Execute and validate

The shared latent-factor runner fits each representation inside its training fold, persists the
fitted state, and requires the complete validation key set before publication.

In [ ]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme_futures-sdf-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

In [ ]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("stochastic discount-factor execution returned a partial prediction")
catalog